# Lesson 4 — Entanglement and Bell States

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 4. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- How H and CNOT create entanglement
- The four Bell states and their circuits
- Measuring correlations: entangled vs separable states
- Bell measurement decoding
- Quantum teleportation with classical corrections
- Superdense coding: two bits over one qubit

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. How Hadamard and CNOT Create Entanglement

A CNOT gate applied to a product state produces another product state. Put the control qubit in superposition with H first, and CNOT entangles the two qubits.

Starting from $|00\rangle$:

$$H|0\rangle \otimes |0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) \otimes |0\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)$$

$$\frac{1}{\sqrt{2}}(|00\rangle + |10\rangle) \xrightarrow{CX} \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

The result cannot be written as $|\psi_0\rangle \otimes |\psi_1\rangle$. The two qubits are **entangled**.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

sv = Statevector.from_label('00').evolve(qc)
print(sv)
print()
print("Probabilities:", sv.probabilities_dict())

In [ ]:
qc.draw('mpl')

Amplitudes at index 0 ($|00\rangle$) and index 3 ($|11\rangle$) are each $1/\sqrt{2}$. Amplitudes at index 1 ($|01\rangle$) and index 2 ($|10\rangle$) are zero. Measurement never produces $|01\rangle$ or $|10\rangle$.

## 2. The Four Bell States

The Bell states form an orthonormal basis for the two-qubit Hilbert space:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$
$$|\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$$
$$|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$$
$$|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$$

All four share the same circuit (H on qubit 0, then CNOT). Initial X gates select which Bell state is produced:

| Initial X gates | Bell state |
|---|---|
| None | $|\Phi^+\rangle$ |
| X on qubit 0 | $|\Phi^-\rangle$ |
| X on qubit 1 | $|\Psi^+\rangle$ |
| X on qubit 0 and qubit 1 | $|\Psi^-\rangle$ |

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def make_bell(x0=False, x1=False):
    """Prepare a Bell state. x0 and x1 select which of the four states."""
    qc = QuantumCircuit(2)
    if x0:
        qc.x(0)
    if x1:
        qc.x(1)
    qc.h(0)
    qc.cx(0, 1)
    return qc

configs = [
    (False, False, '|Φ+⟩'),
    (True,  False, '|Φ-⟩'),
    (False, True,  '|Ψ+⟩'),
    (True,  True,  '|Ψ-⟩'),
]

for x0, x1, label in configs:
    qc = make_bell(x0, x1)
    sv = Statevector.from_label('00').evolve(qc)
    print(f"{label}: {sv.data.round(3)}")

In [ ]:
# Draw all four circuits
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (x0, x1, label) in zip(axes, configs):
    qc = make_bell(x0, x1)
    qc.draw('mpl', ax=ax)
    ax.set_title(label)
plt.tight_layout()
plt.show()

## 3. Measuring Correlations

For $|\Phi^+\rangle$, every time qubit 0 measures $|0\rangle$, qubit 1 is also $|0\rangle$, and vice versa. The outcomes are perfectly correlated. A separable state (H on both qubits independently) shows all four outcomes with equal probability.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

# Entangled: Bell state |Φ+⟩
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])
counts_bell = sim.run(transpile(qc_bell, sim), shots=4096).result().get_counts()

# Separable: H on both qubits (no entanglement)
qc_sep = QuantumCircuit(2, 2)
qc_sep.h(0)
qc_sep.h(1)
qc_sep.measure([0, 1], [0, 1])
counts_sep = sim.run(transpile(qc_sep, sim), shots=4096).result().get_counts()

plot_histogram(
    [counts_bell, counts_sep],
    legend=["Bell state |Φ+⟩", "Separable (H⊗H)|00⟩"],
    title="Entangled vs separable measurement statistics"
)

The Bell state shows only `'00'` and `'11'`; the separable state shows all four outcomes. The correlations in the Bell state cannot be reproduced by any classical pre-shared strategy (Bell's theorem).

### Bell measurement decoding

A Bell measurement decodes a two-qubit state to the computational basis by applying CNOT then H. Each Bell state maps to a unique outcome.

In [ ]:
results = {}
for x0, x1, label in configs:
    qc = QuantumCircuit(2, 2)
    if x0:
        qc.x(0)
    if x1:
        qc.x(1)
    # Prepare Bell state
    qc.h(0)
    qc.cx(0, 1)
    # Decode with inverse Bell circuit
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
    results[label] = counts
    print(f"{label}: {counts}")

Each Bell state decodes to a unique computational basis state with certainty. This is the **Bell measurement** used in both teleportation and superdense coding.

## 4. Quantum Teleportation

Teleportation transfers an arbitrary qubit state from Alice to Bob using a shared Bell pair and two classical bits. The original state is destroyed at Alice's side (no cloning).

**Protocol:**
1. Alice holds the message qubit ($|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$) and her half of $|\Phi^+\rangle$.
2. Alice applies CNOT (message qubit → her Bell qubit), then H on the message qubit, then measures both. She gets 2 classical bits.
3. Alice sends the 2 bits to Bob over a classical channel.
4. Bob applies corrections: X if bit 1 is 1, Z if bit 0 is 1. His qubit is now $|\psi\rangle$.

Qubit layout:
- `q0`: message qubit (Alice)
- `q1`: Alice's half of the Bell pair
- `q2`: Bob's half of the Bell pair

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

theta = np.pi / 3   # arbitrary angle; P(0) = cos²(theta/2) = 0.75

qc = QuantumCircuit(3, 3)

# Prepare message state on q0
qc.ry(theta, 0)
qc.barrier()

# Create shared Bell pair on q1, q2
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# Alice's Bell measurement
qc.cx(0, 1)
qc.h(0)
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()

# Bob's classical corrections
qc.x(2).c_if(1, 1)   # if classical bit 1 == 1, apply X to q2
qc.z(2).c_if(0, 1)   # if classical bit 0 == 1, apply Z to q2

qc.draw('mpl')

### Verification

To verify the teleportation, apply $R_y(-\theta)$ to Bob's qubit after the corrections. If teleportation succeeded, Bob's qubit returns to $|0\rangle$ with certainty regardless of Alice's measurement outcomes.

In [ ]:
qc_verify = QuantumCircuit(3, 3)

# Message state
qc_verify.ry(theta, 0)
qc_verify.barrier()

# Bell pair
qc_verify.h(1)
qc_verify.cx(1, 2)
qc_verify.barrier()

# Bell measurement
qc_verify.cx(0, 1)
qc_verify.h(0)
qc_verify.measure(0, 0)
qc_verify.measure(1, 1)
qc_verify.barrier()

# Corrections
qc_verify.x(2).c_if(1, 1)
qc_verify.z(2).c_if(0, 1)

# Undo the state preparation: if teleportation worked, q2 returns to |0⟩
qc_verify.ry(-theta, 2)
qc_verify.measure(2, 2)

sim = AerSimulator()
counts = sim.run(transpile(qc_verify, sim), shots=1024).result().get_counts()
print("Verification counts:", counts)

# Classical bit 2 is the leftmost character in each bitstring (little-endian)
bob_outcomes = {}
for bitstring, count in counts.items():
    bob_bit = bitstring[0]   # leftmost = classical bit 2
    bob_outcomes[bob_bit] = bob_outcomes.get(bob_bit, 0) + count

print("Bob's qubit:", bob_outcomes)
print("Expected: all outcomes have Bob's bit = '0'")

Alice's two bits (rightmost two characters) vary randomly across the four combinations. Bob's bit (leftmost) is always `0`, confirming the teleportation delivered the correct state in every case.

## 5. Superdense Coding

Superdense coding uses one shared Bell pair to transmit **two classical bits** by sending one qubit. Alice encodes with a local operation; Bob decodes with the inverse Bell circuit.

| Message $(b_0, b_1)$ | Alice's operation | Resulting state |
|---|---|---|
| 00 | $I$ | $|\Phi^+\rangle$ |
| 01 | $X$ | $|\Psi^+\rangle$ |
| 10 | $Z$ | $|\Phi^-\rangle$ |
| 11 | $ZX$ (X then Z) | $|\Psi^-\rangle$ |

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def superdense_coding(b0, b1):
    """
    Build and run the superdense coding circuit for message (b0, b1).
    Returns (counts, circuit).
    """
    qc = QuantumCircuit(2, 2)

    # Setup: create the shared Bell pair
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()

    # Alice's encoding on qubit 0
    if b1 == 1:
        qc.x(0)
    if b0 == 1:
        qc.z(0)
    qc.barrier()

    # Bob's decoding: inverse Bell circuit
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])

    counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
    return counts, qc

# Test all four messages
messages = [(0, 0), (0, 1), (1, 0), (1, 1)]
for b0, b1 in messages:
    counts, _ = superdense_coding(b0, b1)
    print(f"Message ({b0}, {b1}): {counts}")

In [ ]:
# Draw the full circuit for message (1, 1)
_, qc_11 = superdense_coding(1, 1)
qc_11.draw('mpl')

Each two-bit message is recovered with certainty. The shared entanglement allows one qubit of quantum communication to carry two bits of classical information.

### Comparing teleportation and superdense coding

| Protocol | Quantum channel | Classical channel | Effect |
|---|---|---|---|
| Teleportation | 0 qubits sent | 2 bits sent | Transfers 1 qubit state |
| Superdense coding | 1 qubit sent | 0 bits sent | Transfers 2 classical bits |

## 6. Exercises

### Exercise 1: Verifying |Ψ-⟩

The state $|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$ has opposite bits with a relative phase of $-1$. Prepare it with the `make_bell` function and verify:

1. Use `Statevector.probabilities_dict()` to confirm that only `'01'` and `'10'` appear.
2. Apply the inverse Bell circuit (CNOT then H) and measure. Confirm that the outcome is `'11'` with certainty.
3. What makes $|\Psi^-\rangle$ special? It is the only Bell state that is antisymmetric under qubit swap. Verify by checking that swapping qubits 0 and 1 (using `qc.swap(0, 1)`) gives the same state up to a global phase of $-1$.

In [ ]:
# Exercise 1: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

# 1. Prepare |Ψ-⟩ and check probabilities
qc_psi_minus = make_bell(x0=True, x1=True)
sv = Statevector.from_label('00').evolve(qc_psi_minus)
# TODO: print probabilities_dict() and statevector

# 2. Apply inverse Bell and measure
# TODO

# 3. Check antisymmetry under qubit swap
# TODO

### Exercise 2: Teleporting a different state

Teleport the state $|{+}\rangle = H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ instead of $R_y(\pi/3)|0\rangle$.

1. Modify the teleportation circuit to prepare $|{+}\rangle$ on q0 using `qc.h(0)`.
2. For verification, apply H to q2 after the corrections (this maps $|{+}\rangle$ back to $|0\rangle$) and confirm that Bob's bit is always `0`.

In [ ]:
# Exercise 2: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc_ex2 = QuantumCircuit(3, 3)

# TODO: prepare |+⟩ on q0
# TODO: create Bell pair on q1, q2
# TODO: Bell measurement on q0, q1
# TODO: corrections on q2
# TODO: verification — apply H to q2 and measure

sim = AerSimulator()
# TODO: run and check that Bob's bit is always 0

### Exercise 3: Superdense coding with all four messages

Use `plot_histogram` to show the results for all four messages side by side.

1. Run `superdense_coding(b0, b1)` for all four messages and collect the counts.
2. Plot all four histograms in a 1x4 grid using `matplotlib.pyplot.subplots`.
3. Label each subplot with the message sent.

In [ ]:
# Exercise 3: your solution here
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

messages = [(0, 0), (0, 1), (1, 0), (1, 1)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (b0, b1) in zip(axes, messages):
    counts, _ = superdense_coding(b0, b1)
    # TODO: plot the histogram on ax with an appropriate title

plt.suptitle('Superdense Coding: All Four Messages', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 4: Measuring correlations in |Ψ+⟩

The state $|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$ is also maximally entangled, but with opposite-bit correlations instead of same-bit correlations.

1. Prepare $|\Psi^+\rangle$ using `make_bell(x0=False, x1=True)`, add measurements, and run with 4096 shots.
2. Print the counts and verify that only `'01'` and `'10'` appear.
3. Use `plot_histogram` to compare the measurement statistics of $|\Phi^+\rangle$ and $|\Psi^+\rangle$ side by side.

In [ ]:
# Exercise 4: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

# |Φ+⟩ (already computed)
qc_phi = QuantumCircuit(2, 2)
qc_phi.h(0)
qc_phi.cx(0, 1)
qc_phi.measure([0, 1], [0, 1])
counts_phi = sim.run(transpile(qc_phi, sim), shots=4096).result().get_counts()

# TODO: prepare |Ψ+⟩ and run
# TODO: plot both histograms side by side

## Summary

In this lesson you:

- Created entanglement using H followed by CNOT, and verified the result with `Statevector`.
- Prepared all four Bell states by selectively applying X gates before the H and CNOT.
- Observed perfect correlations in Bell state measurements and contrasted them with separable state statistics.
- Decoded Bell states using the inverse Bell circuit (CNOT then H) and verified unique outcomes.
- Implemented quantum teleportation with classical corrections using `c_if` and verified correctness by the inversion trick.
- Implemented superdense coding and confirmed that all four two-bit messages are recovered with certainty.

**Lesson 5** extends to larger systems: tensor products, GHZ and W states, partial measurement, and the swap test.